# ELO Tournament Analysis

This notebook provides tools to analyze and visualize the results of an ELO-based code competition. It supports both **static analysis** of historical runs and **live polling** of active tournaments.

In [ ]:
import os
import json
import pandas as pd
import matplotlib
matplotlib.use('Agg') # Headless support for CI
import matplotlib.pyplot as plt
import glob
import time
from IPython.display import display, clear_output

def find_latest_run(base_dir=".arenas"):
    # Priority 1: Environment Variable (set by test runner)
    env_run = os.environ.get("ELO_RUN_DIR")
    if env_run and os.path.exists(env_run):
        return env_run
        
    # Priority 2: Scan for latest in base_dir
    runs = sorted(glob.glob(os.path.join(base_dir, "elo_run_*")), key=os.path.getmtime, reverse=True)
    if runs: return runs[0]
    
    # Priority 3: Scan for latest in ../base_dir (if running from notebooks/)
    runs = sorted(glob.glob(os.path.join("..", base_dir, "elo_run_*")), key=os.path.getmtime, reverse=True)
    return runs[0] if runs else None

def parse_elo_logs(log_path):
    events = []
    if not os.path.exists(log_path):
        return []
    with open(log_path, "r") as f:
        for line in f:
            if "EVENT_JSON:" in line:
                try:
                    json_str = line.split("EVENT_JSON:")[1].strip()
                    events.append(json.loads(json_str))
                except:
                    continue
    return events

## 1. Static Analysis
Point the notebook to a specific tournament run directory.

In [ ]:
# Configuration
run_dir = find_latest_run()
print(f"Analyzing run directory: {run_dir}")
if run_dir is None:
    raise ValueError("No run directory found!")

log_file = os.path.join(run_dir, "logs", "tournament.log")
events = parse_elo_logs(log_file)
print(f"Parsed {len(events)} events.")

In [ ]:
def process_elo_history(events):
    elo_data = []
    for e in events:
        if e['type'] == 'elo_updated':
            data = e['data']
            label = data['id']
            if data.get('version') is not None:
                label = f"{data['id']} (v{data['version']})"
            
            elo_data.append({
                'timestamp': e['timestamp'],
                'id': label,
                'rating': data['rating'],
                'rd': data['rd'],
                'type': data['type']
            })
    return pd.DataFrame(elo_data)

df_elo = process_elo_history(events)
if not df_elo.empty:
    plt.figure(figsize=(12, 6))
    for label, group in df_elo.groupby('id'):
        plt.plot(group['timestamp'] - df_elo['timestamp'].min(), group['rating'], label=label, marker='o', markersize=4)
    
    plt.title("ELO Rating Trajectories")
    plt.xlabel("Time (seconds since start)")
    plt.ylabel("ELO Rating")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No ELO updates found yet.")

In [ ]:
def show_match_stats(events):
    matches = [e['data'] for e in events if e['type'] == 'match_completed']
    if not matches:
        return pd.DataFrame()
    
    df_matches = pd.DataFrame(matches)
    summary = df_matches.groupby(['candidate_id', 'verifier_id'])['result'].last().unstack()
    return summary

match_matrix = show_match_stats(events)
if not match_matrix.empty:
    print("Recent Match Results (Candidate vs Verifier):")
    display(match_matrix)
else:
    print("No matches completed yet.")

## 2. Live Polling Mode
Run this cell to watch a tournament in progress.

In [ ]:
def live_dashboard(run_path, interval=5, iterations=20):
    log_path = os.path.join(run_path, "logs", "tournament.log")
    
    for _ in range(iterations):
        clear_output(wait=True)
        events = parse_elo_logs(log_path)
        df_elo = process_elo_history(events)
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        if not df_elo.empty:
            for label, group in df_elo.groupby('id'):
                ax1.plot(group['timestamp'] - df_elo['timestamp'].min(), group['rating'], label=label, marker='o', markersize=4)
            ax1.set_title("LIVE: ELO Rating Trajectories")
            ax1.set_ylabel("Rating")
            ax1.legend(loc='center left', bbox_to_anchor=(1, 0.5))
            ax1.grid(True, alpha=0.3)
        
        matches = [e['data'] for e in events if e['type'] == 'match_completed']
        if matches:
            df_m = pd.DataFrame(matches)
            stats = df_m['result'].value_counts()
            stats.plot(kind='bar', ax=ax2, color=['green', 'red', 'orange'])
            ax2.set_title(f"Match Outcomes (Total: {len(matches)})")
            
        plt.tight_layout()
        plt.show()
        
        print(f"Last updated: {time.ctime()} | Events: {len(events)}")
        time.sleep(interval)

# To run live:
# live_dashboard(run_dir)